In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [8]:
import torch
from torch.autograd import Variable
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch import optim
import time
import argparse
import numpy as np
import os
import json
from pathlib import Path
from sys import platform
from datetime import datetime
from torchvision import transforms
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import os
import cv2
import PIL.Image
from functools import partial
import torch.nn.functional as F
from sklearn.metrics import balanced_accuracy_score, precision_recall_fscore_support
import pandas as pd

In [9]:
from argparse import Namespace
args = Namespace(
    run_name="experiment_1",
    input=["mask"],
    n_backbones=1,
    backbone="ResNext18",
    train=True,
    visualize=False,
    test_exp=False,
    viz=False,
    graph=False,
    camera=False,
    tenfps=False,
    conv1_out=8,
    batch_norm=False,
    mask_method="case4",
    mask_prior="none",
    n_epochs=10,
    p_noise=20,
    lr=0.001,
    l2_norm=0.00001,
    gamma=0.5,
    lr_step=20,
    balance_weights=True,
    dataset="raw",
    batch_size=8,
    downscale_factor=8,
    T=16,
    sample_strat="tim",
    threshold=0.6,
    skip=0.5,
    step=1,
    k_fold=0,
    h_flip=False,
    tiny_dataset=False,
    single_run=-1
)

print(f"Start time: {datetime.now()}")
print("\n" * 5, "-"*79, "\n", "-"*79)

# Directory checking
run_path = './runs/' + args.run_name
Path(run_path).mkdir(parents=True, exist_ok=True)

if args.train:
    with open(run_path + '/args.txt', 'w') as f:
        json.dump(args.__dict__, f, indent=2)
        print(f"Saved run parameters to {run_path + '/args.txt'}")

Start time: 2025-04-30 22:10:57.953640





 ------------------------------------------------------------------------------- 
 -------------------------------------------------------------------------------
Saved run parameters to ./runs/experiment_1/args.txt


In [10]:
def get_inplanes():
    return [128, 256, 512, 1024]

class ModifiedBasicBlock(nn.Module):
    expansion = 2

    def __init__(self, in_planes, planes, cardinality, stride=1, downsample=None):
        super().__init__()

        self.conv1 = nn.Conv3d(in_planes,
                               planes,
                               kernel_size=3,
                               stride=stride,
                               padding=1,
                               groups=cardinality,
                               bias=False)
        self.bn1 = nn.BatchNorm3d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv3d(planes, planes * self.expansion, kernel_size=1, bias=False)
        self.bn2 = nn.BatchNorm3d(planes * self.expansion)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out += residual
        out = self.relu(out)

        return out

class ResNeXtBottleneck(nn.Module):
    expansion = 2

    def __init__(self, inplanes, planes, cardinality, stride=1,
                 downsample=None):
        super().__init__()

        mid_planes = cardinality * int(planes / 32)
        self.conv1 = nn.Conv3d(inplanes, mid_planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm3d(mid_planes)
        self.conv2 = nn.Conv3d(
            mid_planes,
            mid_planes,
            kernel_size=3,
            stride=stride,
            padding=1,
            groups=cardinality,
            bias=False)
        self.bn2 = nn.BatchNorm3d(mid_planes)
        self.conv3 = nn.Conv3d(
            mid_planes, planes * self.expansion, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm3d(planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out += residual
        out = self.relu(out)

        return out

class ResNeXt(nn.Module):

    def __init__(self,
                 block,
                 layers,
                 block_inplanes,
                 input_depth,
                 shortcut_type='B',
                 cardinality=32):
        self.inplanes = 64
        super().__init__()
        # block_inplanes = [int(x * widen_factor) for x in block_inplanes]

        self.conv1 = nn.Conv3d(input_depth,
                               self.inplanes,
                               kernel_size=(7),
                               stride=(1, 2, 2),
                               padding=(3, 3, 3),
                               bias=False)
        self.bn1 = nn.BatchNorm3d(self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool3d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, block_inplanes[0], layers[0],
                                       shortcut_type, cardinality)
        self.layer2 = self._make_layer(block,
                                       block_inplanes[1],
                                       layers[1],
                                       shortcut_type,
                                       cardinality,
                                       stride=2)
        self.layer3 = self._make_layer(block,
                                       block_inplanes[2],
                                       layers[2],
                                       shortcut_type,
                                       cardinality,
                                       stride=2)
        self.layer4 = self._make_layer(block,
                                       block_inplanes[3],
                                       layers[3],
                                       shortcut_type,
                                       cardinality,
                                       stride=2)

        self.avgpool = nn.AvgPool3d(
            (1, 4, 4), stride=1)

        # self.avgpool = nn.AdaptiveAvgPool3d(( 1, 1, 1)) # TODO : Compare with avgpool, which was what TIM had.

        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight,
                                        mode='fan_out',
                                        nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _downsample_basic_block(self, x, planes, stride):
        out = F.avg_pool3d(x, kernel_size=1, stride=stride)
        zero_pads = torch.zeros(out.size(0), planes - out.size(1), out.size(2),
                                out.size(3), out.size(4))
        if isinstance(out.data, torch.cuda.FloatTensor):
            zero_pads = zero_pads.cuda()

        out = Variable(torch.cat([out.data, zero_pads], dim=1))

        return out

    def _make_layer(self,
                    block,
                    planes,
                    blocks,
                    shortcut_type,
                    cardinality,
                    stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            if shortcut_type == 'A':
                downsample = partial(
                    self.downsample_basic_block,
                    planes=planes * block.expansion,
                    stride=stride)
            else:
                downsample = nn.Sequential(
                    nn.Conv3d(
                        self.inplanes,
                        planes * block.expansion,
                        kernel_size=1,
                        stride=stride,
                        bias=False), nn.BatchNorm3d(planes * block.expansion))

        layers = []
        layers.append(
            block(self.inplanes, planes, cardinality, stride, downsample))
        self.inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes, cardinality))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)

        x = x.view(x.size(0), -1)
        # print(f"the n_output features: {x.size()}")

        return x

    def resnext18(**kwargs):
        """Constructs a modified bottleneck ResNeXt-18 model.
        """
        model = ResNeXt(ModifiedBasicBlock, [2, 2, 2, 2], get_inplanes(), **kwargs)

        return model

In [11]:
########################################################################################################################################
                                                ######### Get Model ###########
########################################################################################################################################

class OnlyMaskNet(nn.Module):
    def __init__(self, backbone):
        super(OnlyMaskNet, self).__init__()
        print("\nLoading OnlyMaskNet model with only masks as input data")

        self.out_features = 4096    #Last layer size of ResNext and the number of input to the FC layer

        if backbone == 'ResNext101':
            self.model_flow = ResNeXt.resnext101(input_depth=1)

        elif backbone == 'ResNext50':
            self.model_flow = ResNeXt.resnext50(input_depth=1)

        elif backbone == 'ResNext24':
            self.model_flow = ResNeXt.resnext24(input_depth=1)

        elif backbone == 'ResNext18':
            self.model_flow = ResNeXt.resnext18(input_depth=1)

        else:
            raise ValueError(f"Backbone {backbone} is not implemented")
        print(f"Loaded the {backbone} backbone\n\n")

        self.prediction_layer = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(self.out_features, 2),
            nn.Softmax(dim=1)
        )

    def forward(self, mask):
        x = self.model_flow(mask)
        x = self.prediction_layer(x)
        return x


def get_model(args):
    """
    Returns a model based on the selected arguments.
    If CUDA is available, it will load the model onto the GPU.

    Args:
        args: Namespace with the arguments to be used in the model selection. The required arguments are model_inputs,
              n_backbones, backbone.
              model_inputs: must be a list of the inputs to be used by the model (for instance ['flow', 'rgb']).
              n_backbones: must be an int. Contains the number of backbones to be used. Can never be lower than the
                           length of model_inputs (as two backbones on one modality makes no sense)
              backbone: the backbone type which the model should use. Needs to be one of:
                        MobileNetV2, ResNext101, MobileNet2D, ResNext2D, MobileNet-LSTM, ResNext-LSTM.

    Returns:

    """
    model_inputs = args.input
    n_backbones = args.n_backbones
    backbone = args.backbone
    # input_modalities = len(model_inputs)
    # single backbone 3D convolution models
    if 'mask' in model_inputs:
        modalities = model_inputs.copy()
        modalities.remove('mask')
        print('-'*79)
        print(f"!!!!!!!!!!WARNING!!!!!!!!!!\n"
            f"Model will only train/infer on masks!!\n"
            f"!!!!!!!!!!!!!!!!!!!!!!!!!!!")
        print('-' * 79)
        pred_model = OnlyMaskNet(backbone=backbone)
    else:
        raise NotImplementedError("This configuration can't be loaded.")

    if torch.cuda.is_available():
        print("CUDA available: loading the RiskNet model on the GPU")
        pred_model = pred_model.cuda()
    return pred_model

In [12]:
########################################################################################################################################
                                                ######### Object Detections ###########
########################################################################################################################################

target_classes = (0, 1, 2, 3, 5, 7)
combine_pairs = ((0, 1), (0, 3))
combined_classes = ("cyclist", "motorcyclist")
combined_class_ids = (212, 214)

def combine_bboxes(boxes, classes):
    if len(boxes) == 0 or len(classes) == 0:
        return boxes, classes

    remaining_ids = set(range(len(boxes)))
    new_boxes = []
    new_classes = []

    for class_pair, new_class_id in zip(combine_pairs, combined_class_ids):
        ids1 = [i for i in remaining_ids if classes[i] == class_pair[0]]
        ids2 = [i for i in remaining_ids if classes[i] == class_pair[1]]
        if not ids1 or not ids2:
            continue

        iou_matrix = np.zeros((len(ids1), len(ids2)), dtype=float)
        for i1, id1 in enumerate(ids1):
            for i2, id2 in enumerate(ids2):
                l1, t1, r1, b1 = boxes[id1]
                l2, t2, r2, b2 = boxes[id2]
                intersection = max(0, min(r1, r2) - max(l1, l2)) * max(0, min(b1, b2) - max(t1, t2))
                union = (r1-l1)*(b1-t1) + (r2-l2)*(b2-t2) - intersection
                iou_matrix[i1, i2] = intersection / union if union > 0 else 0

        ids1_for_ids2 = np.argmax(iou_matrix, axis=0)
        matched_pairs = [(ids1[ci1], ids2[ci2]) for ci2, ci1 in enumerate(ids1_for_ids2) if iou_matrix[ci1, ci2] > 0.01]

        for i1, i2 in matched_pairs:
            l1, t1, r1, b1 = boxes[i1]
            l2, t2, r2, b2 = boxes[i2]
            new_box = [min(l1, l2), min(t1, t2), max(r1, r2), max(b1, b2)]
            new_boxes.append(new_box)
            new_classes.append(new_class_id)
            remaining_ids.discard(i1)
            remaining_ids.discard(i2)

    # Add remaining boxes and classes
    final_boxes = [boxes[i] for i in remaining_ids] + new_boxes
    final_classes = [classes[i] for i in remaining_ids] + new_classes

    return final_boxes, final_classes

def keep_roadusers_only(boxes, classes):
    boxes_keep = []
    classes_keep = []
    keep_idx = [0, 1, 2, 3, 5, 7]
    for i in range(classes.shape[0]):
        if classes[i] in keep_idx:
            boxes_keep.append(boxes[i, :].tolist())
            classes_keep.append(classes[i])
    return np.array(boxes_keep), np.array(classes_keep)

def predict(image, model, detection_threshold, cutoff_row=250):
    # cutoff_row = int(image.shape[0] * 0.955)
    # image[cutoff_row:, :, :] = 0 #if the bonnet is visible, it will be detected as a car
    # image = normalize_zero_one(image) #normalization doesnt seem to work, YOLO isn't detecting anything

    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    # image = image.transpose((2, 0, 1))
    # print(f"Shape of the image {image.shape}")
    # image = torch.from_numpy(image).to(device)
    # image = image.float().div(255.0).unsqueeze(0)

    t_start = time.time()
    with torch.no_grad():
        output = model(image)
    t_pred = time.time() - t_start
    # print(f"Time taken: {t_pred}")

    ## get all the predicted class names

    # if image is not loaded to cuda
    pred_bboxes = output.pred[0][:, :4].detach().cpu().numpy()
    pred_scores = output.pred[0][:, 4].detach().cpu().numpy()
    pred_classes = output.pred[0][:, 5].detach().cpu().numpy().astype(int)

    # get boxes and classes above the threshold score
    boxes = pred_bboxes[pred_scores >= detection_threshold].astype(np.int32) # size of (x,4) where x is the number of detections and y is Xmin, Ymin, Xmax, Ymax
    classes = pred_classes[pred_scores >= detection_threshold].astype(np.int32)

    return boxes, classes

def detect(frames, model, save_path="detections.json"):
    # If file already exists, load and return
    # if os.path.exists(save_path):
    #     with open(save_path, "r") as f:
    #         detections = json.load(f)
    #     print("Loaded detections from file.")
    #     return detections

    THRESHOLD = 0.5
    detections = []

    for frame in frames:
        # resize the frame to 320 x 180
        frame = cv2.resize(frame, (320, 180))  # resize to the input size of the model

        boxes, classes = predict(frame, model, detection_threshold=THRESHOLD)

        boxes, classes = keep_roadusers_only(boxes, classes)

        classes = classes.tolist()
        boxes = boxes.tolist()
        boxes, classes = combine_bboxes(boxes, classes)

        frame_preds = {
            'Frame': 0,
            'Classes': classes,
            'BBoxes': boxes
        }

        detections.append(frame_preds)

    # # Save to JSON file after processing all frames
    # with open(save_path, "w") as f:
    #     json.dump(detections, f, indent=4)
    # print("Detections saved to file.")

    return detections

In [13]:
########################################################################################################################################
                                                ######### Get Attention Masks ###########
########################################################################################################################################

def denormalize(img, min_sensor_val, max_sensor_val):
    """
    The data is provided in UINT8 values, however the actual sensor values are floats (and can also be negative).
    Therefore, we need to reverse the normalization process which was used to create the UINT8s. The normalization
    process was:
        UINT8 = ((sensor_reading - min_sensor_reading)*255) / (max_sensor_reading - min_sensor_reading)

    Therefore, the original sensor reading is recovered using:
        sensor_reading = (UINT8/255) * (max_sensor_reading - min_sensor_reading) + min_sensor_reading

    Args:
        img: image containing normalized UINT8 values
        min_sensor_val: minimum sensor value.
        max_sensor_val: maximum sensor value

    Returns:
        A denormalized matrix (same dimensions as img) of floating points.
    """
    max_sensor_val = float(max_sensor_val)
    min_sensor_val = float(min_sensor_val)
    out = (img / 255) * (max_sensor_val - min_sensor_val) + min_sensor_val
    return out

def get_masks(detections, img_size, prior_method, divide=True):
    if not divide:
        print("WARNING: NOT DIVIDING BOX COORDINATES! Only allowed in visualization.")

    if prior_method == 'car':
        prior = np.array(PIL.Image.open(
            Path("./priors/gta_prior_car_binary.jpg")))
        prior = denormalize(prior, 0, 1)
    elif prior_method == 'road':
        prior = np.array(PIL.Image.open(
            Path("./priors/gta_prior_road_binary.jpg")))
        prior = denormalize(prior, 0, 1)
    elif prior_method == 'union':
        prior1 = np.array(PIL.Image.open(
            Path("./priors/gta_prior_road_binary.jpg")))
        prior1 = denormalize(prior1, 0, 1)
        prior2 = np.array(PIL.Image.open(
            Path("./priors/gta_prior_car_binary.jpg")))
        prior2 = denormalize(prior2, 0, 1)
        prior = np.logical_or(prior1, prior2).astype(np.uint8)
    else:
        prior = np.ones((120, 160), dtype=np.uint8)

    if prior.shape != (img_size[0], img_size[1]):
        # print(f"Resizing prior shape to fit image size of {img_size}")
        prior = cv2.resize(prior, (img_size[1], img_size[0]))

    threshold = 1440

    num_steps = len(detections)
    run_masks = np.zeros((1, num_steps, img_size[0], img_size[1]))
    for step in range(num_steps):
        frame_detections = detections[step]
        boxes = np.array(frame_detections['BBoxes'])
        classes = np.array(frame_detections['Classes'])
        processed_boxes, processed_classes = filter_boxes(
            boxes, classes, 4, THRESHOLD=threshold)
        mask = masks_from_boxes(
            img_size, processed_boxes, divide_box_coordinates=divide)
        mask = mask * prior
        run_masks[0, step, :, :] = mask
    # plot_masks(run_masks[0, :, :, :])
    return run_masks

def filter_boxes(boxes, classes, case, THRESHOLD=500):  # abblation study:
    num_boxes = boxes.shape[0]
    if num_boxes < 1:
        return boxes, classes
    elif case == 4:
        # Case 3: keep all bounding boxes larger than X
        boxes_keep = []
        classes_keep = []
        for i in range(num_boxes):
            bbox = boxes[i, :]
            dx = bbox[2] - bbox[0]  # x2 - x1
            dy = bbox[3] - bbox[1]  # y2 - y1
            surface = dx * dy
            # print(surface)
            if surface > THRESHOLD:
                boxes_keep.append(bbox)
                classes_keep.append(classes[i])
        return np.array(boxes_keep), np.array(classes_keep)
    else:
        return NotImplementedError("Only cases 1, 2, and 3 have been implemented")

def masks_from_boxes(img_size, boxes, divide_box_coordinates=True):
    """
    important note: the masks are calculated for the ORIGINAL size of the images, aka 360x480. Then resized to new size.
    """
    h_new = img_size[0]
    w_new = img_size[1]
    mask = np.zeros((h_new, w_new))
    num_boxes = boxes.shape[0]
    if num_boxes == 0:
        mask = mask  # nothing changes
    else:
        for i in range(num_boxes):
            if divide_box_coordinates:
                bbox = boxes[i, :] / 2
            else:
                bbox = boxes[i, :]
            dx = bbox[2] - bbox[0]  # x2 - x1
            dy = bbox[3] - bbox[1]  # y2 - y1
            com = (bbox[0] + int(dx/2), bbox[1] + int(dy/2))
            radius = int((min(dx, dy) / 2))  # FOR CHANGING SIZES
            # radius = 5  # FOR FIXED SIZES
            object_mask = create_circular_mask(
                h_new, w_new, center=com, radius=int(radius))
            mask = np.logical_or(mask, object_mask)
    # resize the mask to new size
    mask = mask.astype(np.uint8)
    # mask = cv2.resize(mask, dsize=(w_new, h_new))
    return mask.astype(bool)

def create_circular_mask(h, w, center, radius):
    Y, X = np.ogrid[:h, :w]
    dist_from_center = np.sqrt((X - center[0])**2 + (Y-center[1])**2)

    mask = dist_from_center <= radius
    return mask

In [54]:
########################################################################################################################################
                                                ######### Dataset Prep ###########
########################################################################################################################################
 # Load the Detection model
print(f"Loading the Detection model")
det_model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)
if torch.cuda.is_available():
    print("CUDA available: loading the detection model on the GPU")
    det_model = det_model.cuda()
det_model.eval()

def get_labels(labels_path, idx):
    exists = os.path.exists(labels_path)
    if exists:
        df_labels = pd.read_csv(labels_path)
        # Sort the DataFrame by the first column (filename)
        df_labels = df_labels.sort_values(by=df_labels.columns[0])
        # Reset index after sorting
        df_labels = df_labels.reset_index(drop=True)

        # Use the idx to access the corresponding row
        if idx < len(df_labels):
            run_label = df_labels.iloc[idx, 3]  # Get scalar value for label (column index 3)
            run_toe = df_labels.iloc[idx, 1]    # Get scalar value for time of event (column index 1)
            run_toa = df_labels.iloc[idx, 2]    # Get scalar value for time of alert (column index 2)
            print(f"Retrieved label for index {idx}: label={run_label}")
            return run_label, run_toe, run_toa
        else:
            print(f"Index {idx} out of bounds for CSV with {len(df_labels)} rows")
            return None, None, None
    else:
        print(f"CSV file does not exist: {labels_path}")
        return None, None, None


class DatasetPrep(Dataset):
    """
    The main class for the dataset prep.
    """

    def __init__(self, mode, args, transform=None):
        img_size = [int(720 / args.downscale_factor),
                    int(1280 / args.downscale_factor)]

        self.transform = transform
        inputs = args.input
        h_flip = args.h_flip

        step = args.step

        result_path = Path(f"/root/.cache/kagglehub/competitions/nexar-collision-prediction/{mode}")
        labels_path = Path(f"/root/.cache/kagglehub/competitions/nexar-collision-prediction/{mode}.csv")

        # Sorted ensures compatibility between operating systems
        if os.path.exists(result_path):
            all_results = sorted([x for x in os.listdir(result_path) if x.endswith('.mp4')])
        else:
            all_results = []
            print(f"Warning: Path {result_path} does not exist!")

        # RUNS = [x for x in all_results if x.endswith('.mp4')]

        # initialization
        n_runs = len(all_results)
        counter = 0
        t_start = time.time()
        first_data = True
        for i, RUN in enumerate(all_results):
            frames = []
            counter += 1
            t_run_start = time.time()
            run_video = os.path.join(result_path, all_results[i])
            run_label, toe, toa = get_labels(labels_path, i)
            cap = cv2.VideoCapture(run_video)
            if not cap.isOpened():
                print(f"Error opening video file: {run_video}")
            else:
                # if run_label == 1:
              fps = int(cap.get(cv2.CAP_PROP_FPS)) # get the frames per second of the video
              total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) # get the total number of frames in the video
              # start_frame = int(toa * fps) # start frame (toa = time of alert)
              # end_frame = int(toe * fps) # end frame (toe = time of event)
              frame_idx = 0
              while cap.isOpened():
                  ret, frame = cap.read()
                  if not ret:
                      break
                  if frame_idx % 30 == 0:
                      # frame = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (224, 224))
                      frames.append(frame)
                      if len(frames) >= 8:
                          break

                  # if start_frame + fps >= total_frames:
                  #     if frame_idx > end_frame:
                  #         break
                  #     if frame_idx >= start_frame:
                  #         if (frame_idx - start_frame) % step == 0:
                  #             frames.append(frame)

                  frame_idx += 1
            cap.release()

            detections = detect(frames, model=det_model)

            if 'mask' in inputs:
                run_masks = get_masks(detections, img_size, args.mask_prior)
            else:
                run_masks = np.array([0])

            if first_data:
                self.labels = run_label
                self.masks = run_masks
                first_data = False
            else:
                self.labels = np.hstack((self.labels, run_label))
                self.masks = np.vstack((self.masks, run_masks))
            abc = self.masks.shape
            print(f"Run: {RUN} ({counter}/{n_runs}) \t  Size of run masks: {abc} \t Loaded in: {(time.time() - t_run_start):.0f} s")

        self.n_samples = self.labels.size
        print(f"Loaded data in {(time.time() - t_start)/60:.1f} minutes")
        self.unique, self.counts = np.unique(self.labels, return_counts=True)
        print(f"Unique labels: {self.unique}. Counts: {self.counts}")

    def __getitem__(self, index):
        label = self.labels[index]

        if len(self.masks.shape) > 3:
            mask = self.masks[index]
        else:
            mask = np.zeros_like(np.array([label]))

        if self.transform:
            mask = self.transform(mask)
            mask = self.transform(mask)
            mask = self.transform(mask)

        mask = torch.Tensor(mask)
        # --> 20230926: labels is a numpy array, so not a tensor. Depth would be an easy extension to add with a Transformer
        return mask, label

    def __len__(self):
        return self.n_samples

Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2025-4-30 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)



Loading the Detection model


Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


CUDA available: loading the detection model on the GPU


In [56]:
########################################################################################################################################
                                                ######### Training & val functions ###########
########################################################################################################################################

def get_classification_metrics(prediction, ground_truth):
    if type(ground_truth) == type(list):
        ground_truth = np.array(ground_truth)
    if type(prediction) == type(list):
        prediction = np.array(prediction)

    idx = np.where(ground_truth == -1)
    ground_truth = np.delete(ground_truth, idx)
    prediction = np.delete(prediction, idx)
    balanced_acc = balanced_accuracy_score(ground_truth, prediction)
    precision, recall, fscore, _ = precision_recall_fscore_support(ground_truth, prediction, average='weighted', zero_division=0)
    return balanced_acc, precision, recall, fscore

class AverageMeter(object):
    """Computes and stores the average and current value"""

    def __init__(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

def train_epoch(epoch, data_loader, model, criterion, optimizer, scheduler):
    print("\n" * 1, "-"*79, "\n" * 1)
    print("Epoch {}".format(epoch))
    print("\n" * 1)
    model.train()

    losses = AverageMeter()
    accs = AverageMeter()
    precs = AverageMeter()
    recalls = AverageMeter()
    fscores = AverageMeter()
    all_preds = []
    all_targets = []

    for i, (input_mask, targets) in enumerate(data_loader):
        t1 = time.time()

        if torch.cuda.is_available():
            targets = targets.cuda()
            # input_flow = input_flow.cuda()
            # input_depth = input_depth.cuda()
            # input_rgb = input_rgb.cuda()
            input_mask = input_mask.cuda()

        # input_flow = Variable(input_flow)
        # input_depth = Variable(input_depth)
        # input_rgb = Variable(input_rgb)
        input_mask = Variable(input_mask)
        targets = Variable(targets).type(torch.int64)
        input_mask = input_mask.unsqueeze(1)
        print(f'The size of input: {input_mask.size()}')
        outputs = model(input_mask)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        values, preds = torch.max(outputs, 1) # preds is the index of the max value in the output tensor, since this is a binary classification problem, preds will be 0 or 1
        targets_list = targets.cpu().detach().tolist()
        preds_list = preds.cpu().detach().tolist()
        all_preds.extend(preds_list)
        all_targets.extend(targets_list)

        losses.update(loss.data, len(targets_list))
        bal_acc, precision, recall, fscore = get_classification_metrics(preds_list, targets_list)
        accs.update(bal_acc, len(targets_list))
        precs.update(precision, len(targets_list))
        recalls.update(recall, len(targets_list))
        fscores.update(fscore, len(targets_list))

    print(f"Epoch: [{epoch}][mean]\t Loss: {losses.avg.item():.3f} \t Accuracy (balanced) {accs.avg.item():.3f}\t"
          f"Precision {precs.avg.item():.3f}\t Recall {recalls.avg.item():.3f} \t F-score {fscores.avg.item():.3f}"
          f"\t Average predicted label: {np.mean(np.array(all_preds)):.3f}"
          f"\t Average actual label: {np.mean(np.array(all_targets)):.3f}")

    scheduler.step()

    return fscores.avg.item(), losses.avg.item()

def val_epoch(epoch, data_loader, model, criterion):
    model.eval()

    losses = AverageMeter()
    accs = AverageMeter()
    precs = AverageMeter()
    recalls = AverageMeter()
    fscores = AverageMeter()
    all_preds = []
    all_targets = []

    for i, (input_masks, targets) in enumerate(data_loader):
        t1 = time.time()

        if torch.cuda.is_available():
            targets = targets.cuda()
            input_flow = input_flow.cuda()
            input_depth = input_depth.cuda()
            input_rgb = input_rgb.cuda()
            input_masks = input_masks.cuda()

        with torch.no_grad():
            outputs = model(input_flow, input_depth, input_rgb, input_masks)

        loss = criterion(outputs, targets.type(torch.int64))
        # writer.add_scalar('validation loss', loss.data, epoch * len(data_loader) + i)

        values, preds = torch.max(outputs, 1)
        targets_list = targets.cpu().detach().tolist()
        preds_list = preds.cpu().detach().tolist()
        all_preds.extend(preds_list)
        all_targets.extend(targets_list)

        losses.update(loss.data, len(targets_list))
        bal_acc, precision, recall, fscore = get_classification_metrics(preds_list, targets_list)
        accs.update(bal_acc, len(targets_list))
        precs.update(precision, len(targets_list))
        recalls.update(recall, len(targets_list))
        fscores.update(fscore, len(targets_list))

    print(f"Epoch: [{epoch}][mean]\t Val_Loss: {losses.avg.item():.3f} \t Val_Accuracy (balanced) {accs.avg.item():.3f}\t"
          f"Val_Precision {precs.avg.item():.3f}\t Val_Recall {recalls.avg.item():.3f} \t Val_F-score {fscores.avg.item():.3f}"
          f"\t Average predicted label: {np.mean(np.array(all_preds)):.3f}"
          f"\t Average actual label: {np.mean(np.array(all_targets)):.3f}")

    return fscores.avg.item(), losses.avg.item()

In [57]:
dummy_set = DatasetPrep(mode='train', args=args)
dummy_data_loader = DataLoader(dataset=dummy_set, batch_size=args.batch_size, shuffle=True)

Retrieved label for index 0: label=1


/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.am

Run: 00000.mp4 (1/1500) 	  Size of run masks: (1, 8, 90, 160) 	 Loaded in: 1 s
Retrieved label for index 1: label=1


/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.am

Run: 00003.mp4 (2/1500) 	  Size of run masks: (2, 8, 90, 160) 	 Loaded in: 1 s
Retrieved label for index 2: label=1


...

Run: 00031.mp4 (25/1500) 	  Size of run masks: (25, 8, 90, 160) 	 Loaded in: 1 s
Retrieved label for index 25: label=1


/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.am

Run: 00032.mp4 (26/1500) 	  Size of run masks: (26, 8, 90, 160) 	 Loaded in: 1 s
Retrieved label for index 26: label=1


/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.am

KeyboardInterrupt: 

In [52]:
########################################################################################################################################
                                                ######### Training & Validation loop ###########
########################################################################################################################################

if args.train:
    # Load the model
    model = get_model(args)

    print('-'*79 + "\nLoading training set")
    train_set = DatasetPrep(mode='train', args=args)
    train_data_loader = DataLoader(dataset=train_set, batch_size=args.batch_size, shuffle=True)
    # print_mem_usage()


-------------------------------------------------------------------------------
!!!!!!!!!!WARNING!!!!!!!!!!
Model will only train/infer on masks!!
!!!!!!!!!!!!!!!!!!!!!!!!!!!
-------------------------------------------------------------------------------

Loading OnlyMaskNet model with only masks as input data
Loaded the ResNext18 backbone


CUDA available: loading the RiskNet model on the GPU
-------------------------------------------------------------------------------
Loading training set
Retrieved label for index 0: label=1


/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.am

Run: 00000.mp4 (1/1500) 	  Size of run masks: (1, 8, 90, 160) 	 Loaded in: 1 s
Retrieved label for index 1: label=1


/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.am

Run: 00003.mp4 (2/1500) 	  Size of run masks: (1, 8, 90, 160) 	 Loaded in: 1 s
Retrieved label for index 2: label=1


...

/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.am

Run: 00017.mp4 (13/1500) 	  Size of run masks: (1, 8, 90, 160) 	 Loaded in: 1 s
Retrieved label for index 13: label=1


/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.am

Run: 00018.mp4 (14/1500) 	  Size of run masks: (1, 8, 90, 160) 	 Loaded in: 1 s
Retrieved label for index 14: label=1


KeyboardInterrupt: 

In [58]:

    # print('-'*79 + "\nLoading validation set")
    # val_set = DatasetPrep(mode='val', args=args)
    # val_data_loader = DataLoader(dataset=val_set, batch_size=args.batch_size, shuffle=True)

    weights = torch.Tensor([1., 1.])
    criterion = nn.CrossEntropyLoss(weights)
    if torch.cuda.is_available():
        print("CUDA available: loading the loss function on the GPU")
        criterion.cuda()

    optimizer = optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.l2_norm)
    # scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=args.gamma)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=args.lr_step, gamma=args.gamma)

    best_fscore = 0
    best_loss = 999999
    best_epoch = 0
    t_start_train = time.time()
    train_losses = []
    # val_losses = []
    train_fscores = []
    # val_fscores = []
    for epoch in range(args.n_epochs):
        t_start_epoch = time.time()
        # is_best = False
        fscore, loss = train_epoch(epoch, train_data_loader, model, criterion, optimizer, scheduler)
        save_name = '/checkpoint_' + str(epoch) + '.pth'
        torch.save(model.state_dict(), run_path + save_name)
        print(f"Saved model {save_name}")
        # is_best = loss < best_loss
        print("Epoch {} trained in {:.2f} s".format(epoch, time.time() - t_start_epoch))
        print("\n" * 1)
        # val_fscore, val_loss = val_epoch(epoch, val_data_loader, model, criterion)
        # print("Epoch {} validated in {:.2f} s".format(epoch, time.time() - t_start_epoch))

        train_losses.append(loss)
        # val_losses.append(val_loss)
        train_fscores.append(fscore)
        # val_fscores.append(val_fscore)

        # if (val_fscore > best_fscore):
        #     is_best = True
        #     torch.save(model.state_dict(), run_path + '/checkpoint_' + 'best_epoch.pth')

        # if is_best:
        #     print(f"New best model with Val F-score {val_fscore:.3f}. Previous best Val F-score {best_fscore:.3f}"
        #           f" (epoch {best_epoch})")
        #     best_fscore = val_fscore
        #     # print(f"New best model with loss {val_loss:.3f}. Previous best loss {best_loss:.3f}"
        #     #       f" (epoch {best_epoch})")
        #     # best_loss = val_loss
        #     best_epoch = epoch
        # else:
        #     print(f"Model not better than previous best of F-score {best_fscore:.3f} (epoch {best_epoch})")
            # print(f"Loss {val_loss:.3f} not better than previous best loss {best_loss:.3f} (epoch {best_epoch})")

    print("\n\nTotal train time: {:.1f} minutes".format((time.time() - t_start_train) / 60))

    epochs = list(range(args.n_epochs))
    # Plot the training and validation losses
    plt.figure()
    plt.plot(train_losses, label='Training Loss', marker='o', linestyle='-', color='b')
    # plt.plot(val_losses, label='Validation Loss', marker='o', linestyle='-', color='r')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.xticks(epochs)
    plt.legend()
    plt.title('Training and Validation Loss Over Epochs')
    plt.grid(True)
    plt.savefig(f"{run_path}/loss_plot.png")
    plt.close()

    # Plot the training and validation F-score
    plt.figure()
    plt.plot(train_fscores, label='Training F-score', marker='o', linestyle='-', color='b')
    # plt.plot(val_fscores, label='Validation F-score', marker='o', linestyle='-', color='r')
    plt.xlabel('Epoch')
    plt.ylabel('F-score')
    plt.xticks(epochs)
    plt.legend()
    plt.title('Training and Validation F-score Over Epochs')
    plt.grid(True)
    plt.savefig(f"{run_path}/fscore_plot.png")
    plt.close()

CUDA available: loading the loss function on the GPU

 ------------------------------------------------------------------------------- 

Epoch 0


The size of input: torch.Size([8, 1, 8, 90, 160])


RuntimeError: input image (T: 1 H: 3 W: 5) smaller than kernel size (kT: 1 kH: 4 kW: 4)